> **Public reproducibility copy.** Notebook outputs and execution counts were cleared before public packaging to avoid local-path/account leakage. The scientific code and configuration cells are preserved. See the repository README for required data paths and reproducibility scope.


# 03A — RETFound internal validation and paired analysis

This notebook performs the **R2.2 internal matched analysis** for:

- `RETFOUND_MAE_CFP_FROZEN_PCA128_SVM`;
- canonical `D_VIT_B16_PCA128`;
- canonical `E_FUSION_PCA128_PROPOSED`.

It does **not** use 597-image results and does not require HOG-CNN outputs.

Outputs:

1. prediction integrity audit;
2. recomputed five-seed pooled metrics;
3. mean ± SD summary;
4. fixed seed-42 confusion matrix;
5. class-wise recall by seed;
6. five-seed consensus predictions and metrics;
7. seed-level paired transitions and exact McNemar tests;
8. consensus-level paired transitions;
9. paired bootstrap 95% CIs for metric differences;
10. machine-readable JSON audit and manuscript-ready Markdown summary.

**Metric-label rules**

- Five-seed table: seed-level pooled OOF mean ± sample SD.
- Confusion matrix: fixed seed-42 pooled OOF.
- Consensus: five-seed majority vote; any 2–2–1 tie is resolved by the highest mean predicted probability among tied classes and explicitly counted.
- Paired tests: aligned by the same seed and sample ID, or by the same consensus sample ID.


In [ ]:

from pathlib import Path
import sys, json, zipfile, math, hashlib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    recall_score,
)
from scipy.stats import binomtest

LABELS = ["AMD_DRY", "AMD_WET", "NORMAL"]
SEEDS = [42, 123, 2026, 3407, 7777]
RET_METHOD = "RETFOUND_MAE_CFP_FROZEN_PCA128_SVM"
COMPARATORS = {
    "D_VIT_B16_PCA128": "Frozen ImageNet ViT-PCA128",
    "E_FUSION_PCA128_PROPOSED": "Proposed Hybrid PCA128",
}

def find_package_root(start: Path) -> Path:
    start = start.resolve()
    markers = [
        Path("config") / "AMD_R2_2_BASELINE_CONFIG_LOCK_v1.yaml",
        Path("manifests") / "AMD_DATASET_594_R2_2_MANIFEST.csv",
    ]
    candidates = [start, *start.parents]
    for base in candidates:
        if all((base / marker).exists() for marker in markers):
            return base
    # Search one level below common starting folders.
    for base in candidates[:3]:
        if base.exists():
            for child in base.iterdir():
                if child.is_dir() and all((child / marker).exists() for marker in markers):
                    return child.resolve()
    raise FileNotFoundError(
        "Package root not found. Run Jupyter from the extracted package or set PKG manually."
    )

PKG = find_package_root(Path.cwd())
OUTPUT = PKG / "outputs" / "retfound_paired_analysis"
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Package root:", PKG)
print("Output folder:", OUTPUT)


## Locate the RETFound output, canonical manifest, and nested-CV package

In [ ]:

RET_PATH = PKG / "outputs" / "retfound" / "retfound_internal_sample_predictions.csv"
MANIFEST_PATH = PKG / "manifests" / "AMD_DATASET_594_R2_2_MANIFEST.csv"

zip_names = [
    "AMD_NESTEDCV_MULTISEED_v1(2).zip",
    "AMD_NESTEDCV_MULTISEED_v1.zip",
]
zip_search_roots = [PKG, PKG.parent, Path.cwd(), Path.cwd().parent]
NESTED_ZIP = None
for root in zip_search_roots:
    for name in zip_names:
        candidate = root / name
        if candidate.exists():
            NESTED_ZIP = candidate.resolve()
            break
    if NESTED_ZIP is not None:
        break

assert RET_PATH.exists(), (
    f"RETFound prediction file not found: {RET_PATH}\n"
    "Complete notebook 01 through the internal OOF cell first."
)
assert MANIFEST_PATH.exists(), f"Canonical manifest not found: {MANIFEST_PATH}"
assert NESTED_ZIP is not None, (
    "Canonical nested-CV ZIP was not found. Place AMD_NESTEDCV_MULTISEED_v1(2).zip "
    "in the package folder or one folder above it."
)

print("RETFound predictions:", RET_PATH)
print("Canonical manifest:", MANIFEST_PATH)
print("Canonical nested ZIP:", NESTED_ZIP)


## Load and validate all aligned predictions

In [ ]:

ret = pd.read_csv(RET_PATH)
manifest = pd.read_csv(MANIFEST_PATH)

with zipfile.ZipFile(NESTED_ZIP, "r") as zf:
    prediction_members = [
        name for name in zf.namelist()
        if name.endswith("nested_sample_level_predictions.csv")
    ]
    assert len(prediction_members) == 1, prediction_members
    with zf.open(prediction_members[0]) as f:
        canonical_all = pd.read_csv(f)

canonical = canonical_all[
    canonical_all["scenario"].isin(COMPARATORS.keys())
].copy()

required_pred_cols = {
    "method", "seed", "outer_fold", "sample_id", "source_dataset",
    "true_label", "predicted_label", "correct",
    "prob_AMD_DRY", "prob_AMD_WET", "prob_NORMAL",
}
missing_ret_cols = sorted(required_pred_cols - set(ret.columns))
assert not missing_ret_cols, f"Missing RETFound columns: {missing_ret_cols}"

assert len(manifest) == 594
assert manifest["sample_id"].nunique() == 594
assert len(ret) == 2970, f"Observed {len(ret)} RETFound rows, expected 2970"
assert set(ret["method"].astype(str)) == {RET_METHOD}
assert set(ret["seed"].astype(int)) == set(SEEDS)

key = ["method", "seed", "outer_fold", "sample_id"]
assert not ret.duplicated(key).any(), "Duplicate RETFound prediction keys"

prob_cols = [f"prob_{label}" for label in LABELS]
p = ret[prob_cols].to_numpy(float)
assert np.isfinite(p).all()
assert (p >= -1e-8).all()
assert np.max(np.abs(p.sum(axis=1) - 1.0)) <= 1e-6

expected_correct = (
    ret["true_label"].astype(str) == ret["predicted_label"].astype(str)
).astype(int)
assert np.array_equal(ret["correct"].astype(int).to_numpy(), expected_correct.to_numpy())

for seed in SEEDS:
    g = ret[ret["seed"].astype(int).eq(seed)]
    assert len(g) == 594
    assert g["sample_id"].nunique() == 594
    assert set(g["sample_id"]) == set(manifest["sample_id"])

alignment_cols = ["sample_id", "source_dataset", "true_label", "outer_fold"]
alignment = ret[alignment_cols].drop_duplicates()
assert len(alignment) == 594
m = manifest[alignment_cols].merge(
    alignment,
    on="sample_id",
    suffixes=("_manifest", "_ret"),
    validate="one_to_one",
)
for col in ["source_dataset", "true_label", "outer_fold"]:
    assert (
        m[f"{col}_manifest"].astype(str).to_numpy()
        == m[f"{col}_ret"].astype(str).to_numpy()
    ).all(), f"Manifest mismatch in {col}"

assert set(canonical["scenario"]) == set(COMPARATORS)
assert len(canonical) == 5940
assert not canonical.duplicated(["scenario", "seed", "outer_fold", "sample_id"]).any()

audit = {
    "method": RET_METHOD,
    "expected_rows": 2970,
    "observed_rows": int(len(ret)),
    "unique_sample_ids": int(ret["sample_id"].nunique()),
    "seeds": sorted(ret["seed"].astype(int).unique().tolist()),
    "duplicate_keys": int(ret.duplicated(key).sum()),
    "invalid_probability_rows": int(
        (~(
            np.isfinite(p).all(axis=1)
            & (p >= -1e-8).all(axis=1)
            & (np.abs(p.sum(axis=1) - 1.0) <= 1e-6)
        )).sum()
    ),
    "maximum_probability_sum_error": float(np.max(np.abs(p.sum(axis=1) - 1.0))),
    "inconsistent_correct_flags": int(
        (ret["correct"].astype(int).to_numpy() != expected_correct.to_numpy()).sum()
    ),
    "manifest_alignment_rows": int(len(m)),
    "canonical_comparator_rows": int(len(canonical)),
    "complete_and_valid": True,
}
print(json.dumps(audit, indent=2))


## Recompute seed-level pooled OOF metrics and mean ± SD

In [ ]:

def metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

seed_rows = []
for seed, g in ret.groupby("seed", sort=True):
    rec = metrics(g["true_label"], g["predicted_label"])
    seed_rows.append({
        "method": RET_METHOD,
        "seed": int(seed),
        "n": int(len(g)),
        "pooled_accuracy": rec["accuracy"],
        "pooled_macro_f1": rec["macro_f1"],
        "pooled_balanced_accuracy": rec["balanced_accuracy"],
    })

seed_metrics = pd.DataFrame(seed_rows).sort_values("seed").reset_index(drop=True)

summary_rows = []
for column, label in [
    ("pooled_accuracy", "accuracy"),
    ("pooled_macro_f1", "macro_f1"),
    ("pooled_balanced_accuracy", "balanced_accuracy"),
]:
    values = seed_metrics[column].to_numpy(float)
    summary_rows.append({
        "method": RET_METHOD,
        "metric_aggregation": "five-seed mean of seed-level pooled OOF metrics",
        "metric": label,
        "mean": float(values.mean()),
        "sd_sample": float(values.std(ddof=1)),
        "minimum": float(values.min()),
        "maximum": float(values.max()),
    })

summary = pd.DataFrame(summary_rows)

display(seed_metrics)
display(summary)


## Fixed seed-42 confusion matrix and class-wise recall

In [ ]:

seed42 = ret[ret["seed"].astype(int).eq(42)].copy()
assert len(seed42) == 594

cm = confusion_matrix(
    seed42["true_label"],
    seed42["predicted_label"],
    labels=LABELS,
)
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in LABELS],
    columns=[f"pred_{x}" for x in LABELS],
)

recall_rows = []
for seed, g in ret.groupby("seed", sort=True):
    recalls = recall_score(
        g["true_label"],
        g["predicted_label"],
        labels=LABELS,
        average=None,
        zero_division=0,
    )
    for label, value in zip(LABELS, recalls):
        recall_rows.append({
            "method": RET_METHOD,
            "seed": int(seed),
            "class": label,
            "recall": float(value),
        })

class_recall = pd.DataFrame(recall_rows)
display(cm_df)
display(class_recall.pivot(index="seed", columns="class", values="recall"))


## Five-seed consensus predictions

In [ ]:

def consensus_from_five_seed_predictions(df, method_col, method_value):
    gdf = df[df[method_col].astype(str).eq(method_value)].copy()
    assert len(gdf) == 2970
    rows = []
    tie_count = 0
    for sample_id, g in gdf.groupby("sample_id", sort=True):
        assert len(g) == 5
        true_labels = g["true_label"].astype(str).unique()
        assert len(true_labels) == 1
        votes = g["predicted_label"].astype(str).value_counts()
        max_votes = int(votes.max())
        winners = sorted(votes[votes.eq(max_votes)].index.tolist())
        tie_resolved = len(winners) > 1
        if tie_resolved:
            tie_count += 1
            mean_probs = {
                label: float(g[f"prob_{label}"].mean())
                for label in winners
            }
            predicted = sorted(
                winners,
                key=lambda label: (-mean_probs[label], LABELS.index(label)),
            )[0]
        else:
            predicted = winners[0]
        first = g.iloc[0]
        rows.append({
            "method": method_value,
            "sample_id": sample_id,
            "source_dataset": first["source_dataset"],
            "true_label": true_labels[0],
            "predicted_label": predicted,
            "correct": int(predicted == true_labels[0]),
            "max_vote_count": max_votes,
            "tie_resolved_by_mean_probability": bool(tie_resolved),
            **{
                f"mean_prob_{label}": float(g[f"prob_{label}"].mean())
                for label in LABELS
            },
        })
    result = pd.DataFrame(rows)
    assert len(result) == 594
    assert result["sample_id"].nunique() == 594
    return result, tie_count

ret_consensus, ret_ties = consensus_from_five_seed_predictions(
    ret.rename(columns={"method": "method_key"}),
    "method_key",
    RET_METHOD,
)

canonical_consensus = {}
canonical_ties = {}
for scenario in COMPARATORS:
    temp = canonical[canonical["scenario"].eq(scenario)].copy()
    temp["method_key"] = scenario
    con, ties = consensus_from_five_seed_predictions(temp, "method_key", scenario)
    canonical_consensus[scenario] = con
    canonical_ties[scenario] = ties

ret_consensus_metrics = pd.DataFrame([{
    "method": RET_METHOD,
    "metric_aggregation": "five-seed majority-vote consensus",
    "n": 594,
    **metrics(ret_consensus["true_label"], ret_consensus["predicted_label"]),
    "tie_count_2_2_1": int(ret_ties),
}])

display(ret_consensus_metrics)
print("RETFound consensus ties:", ret_ties)
print("Canonical comparator consensus ties:", canonical_ties)


## Paired transitions and exact McNemar tests

In [ ]:

def exact_mcnemar(a_correct, b_correct):
    a = np.asarray(a_correct, dtype=bool)
    b = np.asarray(b_correct, dtype=bool)
    assert a.shape == b.shape
    a_only = int(np.sum(a & ~b))
    b_only = int(np.sum(~a & b))
    discordant = a_only + b_only
    p_value = (
        1.0 if discordant == 0
        else float(
            binomtest(
                min(a_only, b_only),
                n=discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )
    )
    return {
        "both_correct": int(np.sum(a & b)),
        "both_incorrect": int(np.sum(~a & ~b)),
        "comparator_correct_retfound_wrong": a_only,
        "retfound_correct_comparator_wrong": b_only,
        "retfound_corrected": b_only,
        "retfound_degraded": a_only,
        "net_retfound": b_only - a_only,
        "discordant": discordant,
        "exact_mcnemar_p": p_value,
    }

seed_transition_rows = []
for scenario, comparator_name in COMPARATORS.items():
    old = canonical[canonical["scenario"].eq(scenario)].copy()
    for seed in SEEDS:
        a = old[old["seed"].astype(int).eq(seed)][
            ["sample_id", "true_label", "correct"]
        ].rename(columns={
            "true_label": "true_label_comparator",
            "correct": "comparator_correct",
        })
        b = ret[ret["seed"].astype(int).eq(seed)][
            ["sample_id", "true_label", "correct"]
        ].rename(columns={
            "true_label": "true_label_retfound",
            "correct": "retfound_correct",
        })
        paired = a.merge(b, on="sample_id", validate="one_to_one")
        assert len(paired) == 594
        assert (
            paired["true_label_comparator"].astype(str).to_numpy()
            == paired["true_label_retfound"].astype(str).to_numpy()
        ).all()
        seed_transition_rows.append({
            "comparison": f"{RET_METHOD}_minus_{scenario}",
            "comparator_scenario": scenario,
            "comparator_name": comparator_name,
            "seed": int(seed),
            "n": 594,
            **exact_mcnemar(
                paired["comparator_correct"],
                paired["retfound_correct"],
            ),
        })

seed_transitions = pd.DataFrame(seed_transition_rows)

consensus_transition_rows = []
for scenario, comparator_name in COMPARATORS.items():
    a = canonical_consensus[scenario][
        ["sample_id", "true_label", "correct"]
    ].rename(columns={
        "true_label": "true_label_comparator",
        "correct": "comparator_correct",
    })
    b = ret_consensus[
        ["sample_id", "true_label", "correct"]
    ].rename(columns={
        "true_label": "true_label_retfound",
        "correct": "retfound_correct",
    })
    paired = a.merge(b, on="sample_id", validate="one_to_one")
    assert len(paired) == 594
    consensus_transition_rows.append({
        "comparison": f"{RET_METHOD}_minus_{scenario}",
        "comparator_scenario": scenario,
        "comparator_name": comparator_name,
        "aggregation": "five-seed majority-vote consensus",
        "n": 594,
        **exact_mcnemar(
            paired["comparator_correct"],
            paired["retfound_correct"],
        ),
    })

consensus_transitions = pd.DataFrame(consensus_transition_rows)

display(seed_transitions)
display(consensus_transitions)


## Paired stratified-bootstrap metric-difference CIs

In [ ]:

BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20260730

def stratified_bootstrap_difference(
    comparator_df,
    retfound_df,
    reps=5000,
    seed=20260730,
):
    a = comparator_df[
        ["sample_id", "true_label", "predicted_label"]
    ].rename(columns={"predicted_label": "pred_comparator"})
    b = retfound_df[
        ["sample_id", "true_label", "predicted_label"]
    ].rename(columns={
        "true_label": "true_label_retfound",
        "predicted_label": "pred_retfound",
    })
    paired = a.merge(b, on="sample_id", validate="one_to_one")
    assert len(paired) == 594
    assert (
        paired["true_label"].astype(str).to_numpy()
        == paired["true_label_retfound"].astype(str).to_numpy()
    ).all()

    observed_comparator = metrics(
        paired["true_label"],
        paired["pred_comparator"],
    )
    observed_retfound = metrics(
        paired["true_label"],
        paired["pred_retfound"],
    )
    observed_diff = {
        metric: observed_retfound[metric] - observed_comparator[metric]
        for metric in observed_comparator
    }

    rng = np.random.default_rng(seed)
    class_indices = {
        label: np.flatnonzero(paired["true_label"].astype(str).to_numpy() == label)
        for label in LABELS
    }
    distributions = {metric: np.empty(reps, dtype=float) for metric in observed_diff}

    y_true_all = paired["true_label"].astype(str).to_numpy()
    pred_a_all = paired["pred_comparator"].astype(str).to_numpy()
    pred_b_all = paired["pred_retfound"].astype(str).to_numpy()

    for r in range(reps):
        sampled = np.concatenate([
            rng.choice(indices, size=len(indices), replace=True)
            for indices in class_indices.values()
        ])
        # Shuffle sampled indices only to avoid class-block ordering artifacts.
        rng.shuffle(sampled)
        ma = metrics(y_true_all[sampled], pred_a_all[sampled])
        mb = metrics(y_true_all[sampled], pred_b_all[sampled])
        for metric in distributions:
            distributions[metric][r] = mb[metric] - ma[metric]

    rows = []
    for metric, values in distributions.items():
        rows.append({
            "metric": metric,
            "difference_direction": "RETFound minus comparator",
            "observed_difference": float(observed_diff[metric]),
            "ci95_lower": float(np.quantile(values, 0.025)),
            "ci95_upper": float(np.quantile(values, 0.975)),
            "bootstrap_repetitions": int(reps),
            "bootstrap_seed": int(seed),
            "bootstrap_type": "paired stratified by true class",
            "ci_crosses_zero": bool(
                np.quantile(values, 0.025) <= 0 <= np.quantile(values, 0.975)
            ),
        })
    return pd.DataFrame(rows)

bootstrap_frames = []
for scenario, comparator_name in COMPARATORS.items():
    frame = stratified_bootstrap_difference(
        canonical_consensus[scenario],
        ret_consensus,
        reps=BOOTSTRAP_REPS,
        seed=BOOTSTRAP_SEED,
    )
    frame.insert(0, "comparator_scenario", scenario)
    frame.insert(1, "comparator_name", comparator_name)
    frame.insert(2, "aggregation", "five-seed majority-vote consensus")
    bootstrap_frames.append(frame)

bootstrap_differences = pd.concat(bootstrap_frames, ignore_index=True)
display(bootstrap_differences)


## Save outputs and generate the validation audit

In [ ]:

seed_metrics.to_csv(
    OUTPUT / "retfound_recomputed_seed_level_pooled_metrics.csv",
    index=False,
)
summary.to_csv(
    OUTPUT / "retfound_five_seed_mean_sd_summary.csv",
    index=False,
)
cm_df.to_csv(
    OUTPUT / "retfound_seed42_confusion_matrix.csv",
)
class_recall.to_csv(
    OUTPUT / "retfound_classwise_recall_by_seed.csv",
    index=False,
)
ret_consensus.to_csv(
    OUTPUT / "retfound_five_seed_consensus_predictions.csv",
    index=False,
)
ret_consensus_metrics.to_csv(
    OUTPUT / "retfound_five_seed_consensus_metrics.csv",
    index=False,
)
seed_transitions.to_csv(
    OUTPUT / "retfound_vs_canonical_seed_level_transitions.csv",
    index=False,
)
consensus_transitions.to_csv(
    OUTPUT / "retfound_vs_canonical_consensus_transitions.csv",
    index=False,
)
bootstrap_differences.to_csv(
    OUTPUT / "retfound_vs_canonical_bootstrap_metric_differences.csv",
    index=False,
)

audit.update({
    "retfound_consensus_ties": int(ret_ties),
    "canonical_consensus_ties": {
        key: int(value) for key, value in canonical_ties.items()
    },
    "bootstrap_repetitions": int(BOOTSTRAP_REPS),
    "bootstrap_seed": int(BOOTSTRAP_SEED),
    "paired_comparators": list(COMPARATORS.keys()),
    "holm_status": (
        "Not finalized here. Final Holm correction must include the complete "
        "pre-specified R2.2 comparison family after HOG-CNN and repaired Full-FT "
        "sample predictions are available."
    ),
    "complete_and_valid": True,
})

with open(
    OUTPUT / "RETFOUND_INTERNAL_VALIDATION_AUDIT.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(audit, f, indent=2)

print("Saved outputs to:", OUTPUT)
print("complete_and_valid:", audit["complete_and_valid"])


## Generate manuscript-ready summary

In [ ]:

metric_lookup = summary.set_index("metric")
cons = ret_consensus_metrics.iloc[0]

def pct(value):
    return f"{100 * float(value):.2f}%"

lines = []
lines.append("# RETFound Internal Matched Analysis")
lines.append("")
lines.append("## Validation")
lines.append("")
lines.append(
    f"- Prediction rows: {audit['observed_rows']}/{audit['expected_rows']}."
)
lines.append(f"- Unique images: {audit['unique_sample_ids']}.")
lines.append(f"- Duplicate keys: {audit['duplicate_keys']}.")
lines.append(
    f"- Invalid probability rows: {audit['invalid_probability_rows']}."
)
lines.append(f"- `complete_and_valid = {audit['complete_and_valid']}`.")
lines.append("")
lines.append("## Five-seed pooled OOF")
lines.append("")
for metric in ["accuracy", "macro_f1", "balanced_accuracy"]:
    row = metric_lookup.loc[metric]
    lines.append(
        f"- {metric}: {pct(row['mean'])} ± {pct(row['sd_sample'])} "
        f"(range {pct(row['minimum'])}–{pct(row['maximum'])})."
    )
lines.append("")
lines.append("## Five-seed consensus")
lines.append("")
lines.append(
    f"- Accuracy: {pct(cons['accuracy'])}; macro-F1: {pct(cons['macro_f1'])}; "
    f"balanced accuracy: {pct(cons['balanced_accuracy'])}."
)
lines.append(f"- 2–2–1 ties resolved by mean probability: {ret_ties}.")
lines.append("")
lines.append("## Paired consensus comparisons")
lines.append("")
for row in consensus_transitions.itertuples(index=False):
    lines.append(
        f"- Versus {row.comparator_name}: RETFound corrected {row.retfound_corrected}, "
        f"degraded {row.retfound_degraded}, net {row.net_retfound:+d}, "
        f"exact McNemar p = {row.exact_mcnemar_p:.6g}."
    )
lines.append("")
lines.append("## Bootstrap metric differences")
lines.append("")
for row in bootstrap_differences.itertuples(index=False):
    lines.append(
        f"- RETFound minus {row.comparator_name}, {row.metric}: "
        f"{100 * row.observed_difference:+.2f} pp "
        f"(95% CI {100 * row.ci95_lower:+.2f} to {100 * row.ci95_upper:+.2f} pp); "
        f"CI crosses zero: {row.ci_crosses_zero}."
    )
lines.append("")
lines.append("## Multiplicity note")
lines.append("")
lines.append(
    "Raw exact McNemar p-values are reported here. The final Holm correction is "
    "deferred until the complete pre-specified R2.2 paired-comparison family "
    "includes HOG-CNN and repaired Full-FT sample-level predictions."
)

summary_md = "\n".join(lines)
summary_path = OUTPUT / "RETFOUND_INTERNAL_PAIRED_ANALYSIS_SUMMARY.md"
summary_path.write_text(summary_md, encoding="utf-8")

print(summary_md)
print("\nSummary saved to:", summary_path)
